# Módulo 6: Árboles de Decisión y Ensambles

## Contenido
1. Árboles de decisión
2. Tuning de hiperparámetros
3. Random Forest
4. XGBoost
5. Selección del modelo final

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import xgboost as xgb

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Dataset: scoring de crédito
n = 2000
np.random.seed(42)

df = pd.DataFrame({
    'edad': np.random.randint(18, 65, n),
    'ingreso_mensual': np.random.exponential(3000, n).round(0) + 1500,
    'deuda_actual': np.random.exponential(5000, n).round(0),
    'antiguedad_empleo': np.random.randint(0, 20, n),
    'num_creditos_previos': np.random.poisson(2, n),
    'tipo_vivienda': np.random.choice(['Propia', 'Arrendada', 'Familiar'], n, p=[0.3, 0.5, 0.2]),
    'estado_civil': np.random.choice(['Soltero', 'Casado', 'Divorciado'], n, p=[0.4, 0.45, 0.15]),
})

# Target: default (1) o no (0)
prob_default = (
    0.15
    - 0.003 * df['edad']
    - 0.00005 * df['ingreso_mensual']
    + 0.00003 * df['deuda_actual']
    - 0.01 * df['antiguedad_empleo']
    + 0.03 * df['num_creditos_previos']
    + 0.05 * (df['tipo_vivienda'] == 'Arrendada')
).clip(0.05, 0.7)

df['default'] = (np.random.random(n) < prob_default).astype(int)
print(f"Dataset: {df.shape}, Tasa de default: {df['default'].mean():.2%}")
df.head()

In [ ]:
# Preparar datos
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42)

y_train = df_train.pop('default').values
y_val = df_val.pop('default').values
y_test = df_test.pop('default').values

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
X_test = dv.transform(df_test.to_dict(orient='records'))

feature_names = dv.get_feature_names_out()
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Features: {len(feature_names)}")

## 1. Árboles de Decisión

In [ ]:
# Entrenar un árbol simple
arbol = DecisionTreeClassifier(max_depth=4, random_state=42)
arbol.fit(X_train, y_train)

y_proba = arbol.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_proba)
print(f"Árbol (depth=4) AUC: {auc:.4f}")

# Visualizar
plt.figure(figsize=(20, 8))
plot_tree(arbol, feature_names=feature_names,
          class_names=['No default', 'Default'],
          filled=True, rounded=True, max_depth=3, fontsize=8)
plt.title('Árbol de Decisión (primeros 3 niveles)')
plt.tight_layout()
plt.show()

## 2. Tuning de hiperparámetros

In [ ]:
# Buscar mejor profundidad
train_aucs, val_aucs = [], []
depths = range(1, 20)

for d in depths:
    arbol = DecisionTreeClassifier(max_depth=d, random_state=42)
    arbol.fit(X_train, y_train)
    train_aucs.append(roc_auc_score(y_train, arbol.predict_proba(X_train)[:, 1]))
    val_aucs.append(roc_auc_score(y_val, arbol.predict_proba(X_val)[:, 1]))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_aucs, 'b-o', label='Train', markersize=4)
plt.plot(depths, val_aucs, 'r-o', label='Validación', markersize=4)
plt.xlabel('max_depth')
plt.ylabel('AUC')
plt.title('Overfitting: AUC en Train vs Validación')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

mejor_depth = depths[np.argmax(val_aucs)]
print(f"Mejor depth: {mejor_depth} (AUC={max(val_aucs):.4f})")

## 3. Random Forest

In [ ]:
# Random Forest con diferentes configuraciones
configs = [
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 6},
    {'n_estimators': 200, 'max_depth': 7},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 200, 'max_depth': 5},
]

print(f"{'Config':<35} {'AUC Val':>8}")
print("-" * 45)

mejor_auc_rf = 0
mejor_rf = None

for cfg in configs:
    rf = RandomForestClassifier(**cfg, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    auc = roc_auc_score(y_val, rf.predict_proba(X_val)[:, 1])
    print(f"n={cfg['n_estimators']}, depth={cfg['max_depth']:<15} {auc:>8.4f}")
    if auc > mejor_auc_rf:
        mejor_auc_rf = auc
        mejor_rf = rf

In [ ]:
# Feature importance del Random Forest
importancia = pd.DataFrame({
    'feature': feature_names,
    'importancia': mejor_rf.feature_importances_
}).sort_values('importancia', ascending=False)

plt.figure(figsize=(10, 5))
top10 = importancia.head(10)
plt.barh(top10['feature'], top10['importancia'], color='steelblue')
plt.xlabel('Importancia')
plt.title('Top 10 Features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. XGBoost

In [ ]:
# Entrenar XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_names))
dval = xgb.DMatrix(X_val, label=y_val, feature_names=list(feature_names))

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 4,
    'learning_rate': 0.1,
    'verbosity': 0,
}

modelo_xgb = xgb.train(
    params, dtrain,
    num_boost_round=200,
    evals=[(dval, 'val')],
    early_stopping_rounds=20,
    verbose_eval=20
)

y_proba_xgb = modelo_xgb.predict(dval)
auc_xgb = roc_auc_score(y_val, y_proba_xgb)
print(f"\nXGBoost AUC: {auc_xgb:.4f} (mejor iteración: {modelo_xgb.best_iteration})")

In [ ]:
# Tuning de XGBoost
resultados = []

for depth in [3, 4, 5, 6]:
    for lr in [0.05, 0.1, 0.2]:
        params = {
            'objective': 'binary:logistic',
            'eval_metric': 'auc',
            'max_depth': depth,
            'learning_rate': lr,
            'verbosity': 0,
        }
        mod = xgb.train(
            params, dtrain,
            num_boost_round=200,
            evals=[(dval, 'val')],
            early_stopping_rounds=20,
            verbose_eval=False
        )
        auc = roc_auc_score(y_val, mod.predict(dval))
        resultados.append((auc, depth, lr, mod.best_iteration))

resultados.sort(reverse=True)
print(f"{'AUC':<8} {'Depth':<7} {'LR':<6} {'Trees':<6}")
print("-" * 30)
for auc, depth, lr, trees in resultados[:5]:
    print(f"{auc:<8.4f} {depth:<7} {lr:<6} {trees:<6}")

## 5. Selección del modelo final

In [ ]:
# Comparar todos los modelos
# Logística
lr_model = LogisticRegression(solver='liblinear', max_iter=1000)
lr_model.fit(X_train, y_train)
auc_lr = roc_auc_score(y_val, lr_model.predict_proba(X_val)[:, 1])

# Mejor árbol
best_tree = DecisionTreeClassifier(max_depth=mejor_depth, random_state=42)
best_tree.fit(X_train, y_train)
auc_tree = roc_auc_score(y_val, best_tree.predict_proba(X_val)[:, 1])

print("\n" + "=" * 40)
print("COMPARACIÓN FINAL")
print("=" * 40)
print(f"{'Modelo':<20} {'AUC Val':>8}")
print("-" * 30)
print(f"{'Logística':<20} {auc_lr:>8.4f}")
print(f"{'Árbol':<20} {auc_tree:>8.4f}")
print(f"{'Random Forest':<20} {mejor_auc_rf:>8.4f}")
print(f"{'XGBoost':<20} {auc_xgb:>8.4f}")

In [ ]:
# Modelo final: entrenar con train+val, evaluar en test
X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

# Usar la mejor configuración de XGBoost
best_auc, best_depth, best_lr, best_trees = resultados[0]

dfull = xgb.DMatrix(X_full, label=y_full, feature_names=list(feature_names))
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=list(feature_names))

params_final = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': best_depth,
    'learning_rate': best_lr,
    'verbosity': 0,
}

modelo_final = xgb.train(params_final, dfull, num_boost_round=best_trees)
auc_test = roc_auc_score(y_test, modelo_final.predict(dtest))

print(f"\nAUC FINAL en TEST: {auc_test:.4f}")
print(f"Config: depth={best_depth}, lr={best_lr}, trees={best_trees}")

In [ ]:
# Guardar el modelo
import joblib

modelo_final.save_model('../modelo_credito.json')
joblib.dump(dv, '../vectorizer.joblib')
print("Modelo guardado: modelo_credito.json")
print("Vectorizer guardado: vectorizer.joblib")